# Player Action Value & Possession Threat Explorer

**A notebook-first football analytics case study focused on how individual actions improve possession value and increase threat.**

This project uses processed StatsBomb-based action data to examine which players create value, which actions are doing the work behind that value, and how teams differ in the way they move possessions into dangerous areas. The notebook is written as a practical reporting layer: clear questions, stable filters, readable visuals, and interpretation grounded in football use rather than model exposition.

**Main questions**

- Which players generate the most total expected threat across the sample?
- Which players create value most efficiently once playing time is controlled?
- Which actions contribute most strongly to possession value?
- How do teams differ between earlier progression threat and later final-third threat?
- Which players look valuable even without dominating action volume?
- What does player style clustering add beyond standard leaderboards?


## Data Used in This Project

### A. Raw source data

The original source is **StatsBomb Open Data**. In simple terms, this is event data: a record of what happened on the ball during a match, including passes, carries, dribbles, shots, duels, recoveries, and other actions, along with timing and pitch location.

That type of data is the right input for a possession-value project because it captures *how* attacks develop. Goals and assists tell us what finished a move. Event data helps explain what improved the move earlier in the sequence.

### B. Processed data used directly in the notebook

The notebook reads four processed parquet files from `data/processed/`.

- `actions.parquet`: action-level data used to study how value is created inside possessions. Key fields include `player_name`, `team_name`, `action_type`, `xt_added`, `progression_xt`, and `final_third_threat`. This file is the foundation for action-type analysis and the player-style feature engineering used in the ML section.
- `player_summary.parquet`: player-level aggregated output used for leaderboards and profile views. Important fields include `player_name`, `team_name`, `minutes_played`, `action_count`, `total_xt`, and `xt_per90`. This is the main source for player comparison.
- `team_summary.parquet`: team-level aggregated output used to compare team threat generation. Important fields include `team_name`, `matches_played`, `total_xt`, `progression_xt`, `final_third_threat`, and `xt_per_match`. This supports the team-style section.
- `action_type_summary.parquet`: action-type summary output used to show which actions contribute the most xT in aggregate. Important fields include `type_name`, `actions`, `xt_total`, and `xt_mean`.

### C. Transformation pipeline

The underlying pipeline is straightforward.

1. StatsBomb event data is loaded from the raw source.
2. Events are converted into action-level representations that are easier to use for possession analysis.
3. xT values are assigned using action start and end locations.
4. Action-level values are aggregated into player, team, and action-type summaries.
5. This notebook reads the processed parquet outputs for analysis and visualization.

### D. Why these data choices make sense

xT is useful here because it values buildup and progression, not just final outcomes. Action-level data is necessary because possession value is created step by step rather than appearing only at the shot. Player and team summaries make comparison easier, while the processed parquet outputs keep the notebook fast, reproducible, and focused on analysis instead of recomputing the full pipeline live.


## Data Source and Methodology

**StatsBomb Open Data** is well suited to this project because it combines broad event coverage with location data detailed enough to support possession-value work. That makes it possible to move beyond headline outcomes and examine how attacks improve over multiple actions.

**SPADL-style action framing** is useful because it gives the event feed a consistent possession language. Passes, dribbles, shots, and other actions can then be valued on the same scale instead of treated as unrelated event types.

**Expected threat (xT)** is the right first model for this notebook because it is practical and interpretable. It estimates how much an action improves a possession by moving the ball into a more dangerous area. That makes it especially useful for identifying buildup value, progression quality, and stylistic differences between players and teams.

The goal is not to claim that xT captures everything. It is to build a clear first action-value layer that answers a football question directly: who is improving the possession, and what type of actions are driving that improvement?


## Data Loading

The notebook reads the processed parquet outputs directly with `pathlib` and keeps the setup intentionally light. If a file is missing or a required column is unavailable, the notebook flags that clearly so the analysis does not fail silently.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.options.display.float_format = "{:.3f}".format

candidate_roots = [Path.cwd(), Path.cwd().parent]
repo_root = next(
    (path for path in candidate_roots if (path / "data" / "processed").exists() and (path / "src").exists()),
    Path.cwd(),
)
processed_dir = repo_root / "data" / "processed"


def load_parquet(name, required_columns):
    path = processed_dir / name
    if not path.exists():
        return None, f"Missing file: {path}"

    df = pd.read_parquet(path)
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        return None, f"{name} is missing required columns: {', '.join(missing)}"

    return df.copy(), None


def coerce_numeric(df, columns):
    if df is None:
        return None
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, pd.NA)
    return numerator / denominator


player_summary, player_error = load_parquet(
    "player_summary.parquet",
    ["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"],
)
team_summary, team_error = load_parquet(
    "team_summary.parquet",
    ["team_name", "matches_played", "total_xt", "progression_xt", "final_third_threat", "xt_per_match", "progressive_actions"],
)
action_type_summary, action_type_error = load_parquet(
    "action_type_summary.parquet",
    ["type_name", "actions", "xt_total", "xt_mean"],
)
actions, actions_error = load_parquet(
    "actions.parquet",
    ["player_name", "team_name", "action_type", "xt_added", "progression_xt", "final_third_threat"],
)

player_summary = coerce_numeric(player_summary, ["minutes_played", "action_count", "total_xt", "xt_per90", "xt_per_90"])
team_summary = coerce_numeric(team_summary, ["matches_played", "total_xt", "progression_xt", "final_third_threat", "xt_per_match", "progressive_actions"])
action_type_summary = coerce_numeric(action_type_summary, ["actions", "xt_total", "xt_mean", "xt_positive", "xt_negative", "avg_xt"])
actions = coerce_numeric(actions, ["xt_added", "progression_xt", "final_third_threat", "xT_start", "xT_end", "xt_start", "xt_end"])

status_rows = [
    {"dataset": "actions.parquet", "level": "action", "available": actions is not None, "detail": actions_error or f"{len(actions):,} rows"},
    {"dataset": "player_summary.parquet", "level": "player", "available": player_summary is not None, "detail": player_error or f"{len(player_summary):,} rows"},
    {"dataset": "team_summary.parquet", "level": "team", "available": team_summary is not None, "detail": team_error or f"{len(team_summary):,} rows"},
    {"dataset": "action_type_summary.parquet", "level": "action type", "available": action_type_summary is not None, "detail": action_type_error or f"{len(action_type_summary):,} rows"},
]

display(pd.DataFrame(status_rows))

if any(message is not None for message in [player_error, team_error, action_type_error, actions_error]):
    display(Markdown("Run `python -u scripts/run_pipeline.py` from the repository root if the processed parquet outputs need to be rebuilt."))


,dataset,level,available,detail
0,actions.parquet,action,True,"128,484 rows"
1,player_summary.parquet,player,True,603 rows
2,team_summary.parquet,team,True,32 rows
3,action_type_summary.parquet,action type,True,21 rows


In [2]:
MIN_PLAYER_MINUTES = 180
RATE_MINUTES = 360
LOW_VOLUME_MINUTES = 180
LOW_VOLUME_ACTIONS_MIN = 50
LOW_VOLUME_ACTIONS_MAX = 300

qualified_players = None
rate_players = None
low_volume_players = None
top_player_table = None
total_xt_chart_df = None
per90_chart_df = None
action_type_chart_df = None
team_scatter_df = None
team_bar_df = None
low_volume_table = None

if player_summary is not None:
    qualified_players = player_summary.loc[
        (player_summary["minutes_played"].fillna(0) >= MIN_PLAYER_MINUTES)
        & player_summary[["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"]].notna().all(axis=1)
    ].copy()

    rate_players = player_summary.loc[
        (player_summary["minutes_played"].fillna(0) >= RATE_MINUTES)
        & player_summary[["player_name", "team_name", "minutes_played", "total_xt", "xt_per90"]].notna().all(axis=1)
    ].copy()

    low_volume_players = player_summary.loc[
        (player_summary["minutes_played"].fillna(0) >= LOW_VOLUME_MINUTES)
        & (player_summary["action_count"].fillna(0) >= LOW_VOLUME_ACTIONS_MIN)
        & (player_summary["action_count"].fillna(0) <= LOW_VOLUME_ACTIONS_MAX)
        & player_summary[["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"]].notna().all(axis=1)
    ].copy()

    top_player_table = (
        qualified_players.loc[:, ["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"]]
        .sort_values(["total_xt", "xt_per90"], ascending=[False, False])
        .head(12)
        .rename(
            columns={
                "player_name": "Player",
                "team_name": "Team",
                "minutes_played": "Minutes",
                "action_count": "Actions",
                "total_xt": "Total xT",
                "xt_per90": "xT per 90",
            }
        )
    )

    total_xt_chart_df = (
        qualified_players.loc[:, ["player_name", "team_name", "minutes_played", "total_xt"]]
        .sort_values("total_xt", ascending=False)
        .head(15)
        .sort_values("total_xt", ascending=True)
    )

    per90_chart_df = (
        rate_players.loc[:, ["player_name", "team_name", "minutes_played", "total_xt", "xt_per90"]]
        .sort_values("xt_per90", ascending=False)
        .head(15)
        .sort_values("xt_per90", ascending=True)
    )

    low_volume_table = (
        low_volume_players.loc[:, ["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"]]
        .sort_values(["xt_per90", "total_xt"], ascending=[False, False])
        .head(10)
        .rename(
            columns={
                "player_name": "Player",
                "team_name": "Team",
                "minutes_played": "Minutes",
                "action_count": "Actions",
                "total_xt": "Total xT",
                "xt_per90": "xT per 90",
            }
        )
    )

if action_type_summary is not None:
    action_type_chart_df = (
        action_type_summary.loc[:, [column for column in ["type_name", "actions", "xt_total", "xt_mean"] if column in action_type_summary.columns]]
        .dropna(subset=["xt_total"])
        .sort_values("xt_total", ascending=False)
    )

if team_summary is not None:
    team_scatter_df = (
        team_summary.loc[:, ["team_name", "progression_xt", "final_third_threat", "total_xt", "matches_played", "xt_per_match"]]
        .dropna()
    )
    team_bar_df = (
        team_summary.loc[:, ["team_name", "xt_per_match", "total_xt", "matches_played"]]
        .dropna(subset=["xt_per_match"])
        .sort_values("xt_per_match", ascending=False)
    )


## Player Value Exploration

The player section starts with a stable playing-time filter so the leaderboard reflects repeatable contribution rather than short bursts of efficiency. The goal is to separate two related but different ideas: sustained value over volume, and efficient value on a per-minute basis.


In [3]:
if top_player_table is None or top_player_table.empty:
    display(Markdown(player_error or "No player rows are available for the leaderboard."))
else:
    table = top_player_table.copy()
    table["Minutes"] = table["Minutes"].round(0).astype(int)
    table["Actions"] = table["Actions"].round(0).astype(int)
    table["Total xT"] = table["Total xT"].round(2)
    table["xT per 90"] = table["xT per 90"].round(2)
    display(table)

    leader = table.iloc[0]
    display(Markdown(
        f"This leaderboard gives a stable first read on player contribution by keeping a basic minutes floor in place. "
        f"**{leader['Player']}** currently leads the qualified sample, which points to sustained possession value rather than a short-run spike. "
        "Keeping both total xT and xT per 90 in view helps separate players who drive value through repeated involvement from those whose impact is more rate-driven."
    ))


,Player,Team,Minutes,Actions,Total xT,xT per 90
0,Kieran Trippier,England,266,684,2.670,0.900
2,Luka Modrić,Croatia,447,1033,2.430,0.490
3,Kevin De Bruyne,Belgium,232,714,1.910,0.740
4,Antoine Griezmann,France,557,542,1.850,0.300
5,Ivan Rakitić,Croatia,375,833,1.780,0.430
7,Ivan Perišić,Croatia,372,446,1.760,0.430
13,Willian Borges da Silva,Brazil,181,433,1.550,0.770
14,Philippe Coutinho Correia,Brazil,259,696,1.500,0.520
16,Eden Hazard,Belgium,223,687,1.440,0.580
17,Dries Mertens,Belgium,232,273,1.270,0.490


This leaderboard gives a stable first read on player contribution by keeping a basic minutes floor in place. **Kieran Trippier** currently leads the qualified sample, which points to sustained possession value rather than a short-run spike. Keeping both total xT and xT per 90 in view helps separate players who drive value through repeated involvement from those whose impact is more rate-driven.

In [4]:
if total_xt_chart_df is None or total_xt_chart_df.empty:
    display(Markdown(player_error or "No player rows are available for the total xT chart."))
else:
    fig = px.bar(
        total_xt_chart_df,
        x="total_xt",
        y="player_name",
        orientation="h",
        color_discrete_sequence=["#0f5c4d"],
        text="total_xt",
        hover_name="player_name",
        hover_data={"team_name": True, "minutes_played": True, "total_xt": ":.2f"},
        title="Top Players by Total xT",
        template="plotly_white",
    )
    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig.update_layout(height=540)
    fig.update_xaxes(title="Total xT")
    fig.update_yaxes(title=None)
    display(fig)

    top_row = total_xt_chart_df.sort_values("total_xt", ascending=False).iloc[0]
    display(Markdown(
        f"This chart ranks the strongest aggregate possession-value contributors in the sample. "
        f"**{top_row['player_name']}** leads on total xT, which suggests repeated value creation across a meaningful action load. "
        "For player evaluation, this is the clearest first pass on who is consistently moving attacks into better spaces over time."
    ))


This chart ranks the strongest aggregate possession-value contributors in the sample. **Kieran Trippier** leads on total xT, which suggests repeated value creation across a meaningful action load. For player evaluation, this is the clearest first pass on who is consistently moving attacks into better spaces over time.

In [5]:
if per90_chart_df is None or per90_chart_df.empty:
    display(Markdown(player_error or "No player rows are available for the xT per 90 chart."))
else:
    fig = px.bar(
        per90_chart_df,
        x="xt_per90",
        y="player_name",
        orientation="h",
        color_discrete_sequence=["#145a7a"],
        text="xt_per90",
        hover_name="player_name",
        hover_data={"team_name": True, "minutes_played": True, "total_xt": ":.2f", "xt_per90": ":.2f"},
        title="Top Players by xT per 90 (360+ Minutes)",
        template="plotly_white",
    )
    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
    fig.update_layout(height=540)
    fig.update_xaxes(title="xT per 90")
    fig.update_yaxes(title=None)
    display(fig)

    rate_leader = per90_chart_df.sort_values("xt_per90", ascending=False).iloc[0]
    display(Markdown(
        f"The rate view applies a firmer minutes threshold so the ranking is not dominated by short appearances. "
        f"**{rate_leader['player_name']}** leads the qualified group on xT per 90, which points to a stronger efficiency profile when on the pitch. "
        "Read alongside total xT, it helps distinguish players whose value comes through scale from those whose value is concentrated into fewer minutes."
    ))


The rate view applies a firmer minutes threshold so the ranking is not dominated by short appearances. **Luka Modrić** leads the qualified group on xT per 90, which points to a stronger efficiency profile when on the pitch. Read alongside total xT, it helps distinguish players whose value comes through scale from those whose value is concentrated into fewer minutes.

## Action Type Analysis

Player rankings show *who* is creating possession value. The action-type view shifts the focus to *how* that value is being generated inside possessions.


In [6]:
if action_type_chart_df is None or action_type_chart_df.empty:
    display(Markdown(action_type_error or "No action-type rows are available for charting."))
else:
    fig = px.bar(
        action_type_chart_df,
        x="type_name",
        y="xt_total",
        color="xt_total",
        color_continuous_scale="Tealgrn",
        hover_name="type_name",
        hover_data={"actions": True, "xt_total": ":.2f", "xt_mean": ":.4f"},
        title="xT by Action Type",
        template="plotly_white",
    )
    fig.update_layout(height=500)
    fig.update_xaxes(title="Action Type", categoryorder="total descending")
    fig.update_yaxes(title="Total xT")
    display(fig)

    top_three = action_type_chart_df.head(3)["type_name"].tolist()
    if len(top_three) == 3:
        display(Markdown(
            f"The action mix is led by **{top_three[0]}**, followed by **{top_three[1]}** and **{top_three[2]}**. "
            "That matters because possession value is not created evenly across all touches. "
            "This view helps connect player output back to the specific action types that are improving field position and carrying attacks into stronger areas."
        ))


The action mix is led by **shot**, followed by **cross** and **pass**. That matters because possession value is not created evenly across all touches. This view helps connect player output back to the specific action types that are improving field position and carrying attacks into stronger areas.

## Team Style Analysis

The team section shifts from individual contribution to possession structure. The goal is not just to rank threat output, but to show whether teams are creating value earlier through progression or later once attacks reach advanced zones.


In [7]:
if team_scatter_df is None or team_scatter_df.empty:
    display(Markdown(team_error or "No team rows are available for the progression-versus-final-third chart."))
else:
    fig = px.scatter(
        team_scatter_df,
        x="progression_xt",
        y="final_third_threat",
        size="matches_played",
        color="xt_per_match",
        text="team_name",
        hover_name="team_name",
        hover_data={
            "total_xt": ":.2f",
            "xt_per_match": ":.2f",
            "matches_played": True,
            "progression_xt": ":.2f",
            "final_third_threat": ":.2f",
        },
        title="Team Progression xT vs Final-Third Threat",
        template="plotly_white",
        color_continuous_scale="Tealgrn",
    )
    fig.update_layout(height=560)
    fig.update_xaxes(title="Progression xT")
    fig.update_yaxes(title="Final-Third Threat")
    display(fig)

    top_progression = team_scatter_df.sort_values("progression_xt", ascending=False).iloc[0]
    top_final_third = team_scatter_df.sort_values("final_third_threat", ascending=False).iloc[0]
    display(Markdown(
        f"The scatter separates where teams are generating threat inside the possession. "
        f"**{top_progression['team_name']}** leads the sample on progression xT, while **{top_final_third['team_name']}** leads on final-third threat. "
        "That split is useful because teams can arrive at similar attacking output through very different possession structures and field-position patterns."
    ))


The scatter separates where teams are generating threat inside the possession. **England** leads the sample on progression xT, while **Brazil** leads on final-third threat. That split is useful because teams can arrive at similar attacking output through very different possession structures and field-position patterns.

In [8]:
if team_bar_df is None or team_bar_df.empty:
    display(Markdown(team_error or "No team rows are available for the xT per match chart."))
else:
    fig = px.bar(
        team_bar_df,
        x="team_name",
        y="xt_per_match",
        color="xt_per_match",
        color_continuous_scale="Tealgrn",
        hover_name="team_name",
        hover_data={"total_xt": ":.2f", "matches_played": True, "xt_per_match": ":.2f"},
        title="Teams by xT per Match",
        template="plotly_white",
    )
    fig.update_layout(height=520)
    fig.update_xaxes(title="Team", categoryorder="total descending")
    fig.update_yaxes(title="xT per Match")
    display(fig)

    top_team = team_bar_df.iloc[0]
    display(Markdown(
        f"This chart compresses team threat into a game-by-game rate and gives a cleaner top-line comparison than raw totals alone. "
        f"**{top_team['team_name']}** currently leads the sample on xT per match, which suggests the strongest overall possession-value output on a per-game basis. "
        "Used with the scatter above, it helps separate total threat level from the route a team uses to create it."
    ))


This chart compresses team threat into a game-by-game rate and gives a cleaner top-line comparison than raw totals alone. **Germany** currently leads the sample on xT per match, which suggests the strongest overall possession-value output on a per-game basis. Used with the scatter above, it helps separate total threat level from the route a team uses to create it.

## Low-Volume, High-Value Profiles

This section is designed to catch players who may be missed by broad volume-led rankings. The filter is intentionally conservative: players must clear a basic minutes threshold, sit inside a sensible action-count band, and retain a valid xT per 90 value.

That matters because very small samples can produce inflated per-90 numbers that are not analytically useful. The aim here is to surface realistic lower-volume profiles rather than short-run noise.


In [9]:
if low_volume_table is None or low_volume_table.empty:
    display(Markdown(player_error or "No players meet the low-volume, high-value profile filters."))
else:
    table = low_volume_table.copy()
    table["Minutes"] = table["Minutes"].round(0).astype(int)
    table["Actions"] = table["Actions"].round(0).astype(int)
    table["Total xT"] = table["Total xT"].round(2)
    table["xT per 90"] = table["xT per 90"].round(2)
    display(table)

    standout = table.iloc[0]
    display(Markdown(
        f"This view looks for players who add clear possession value without dominating action volume. "
        f"Under the current filters, **{standout['Player']}** is the strongest low-volume profile in the sample. "
        "That makes the section useful for identifying efficient contributors who might be missed by broader leaderboards built mainly on total output."
    ))


,Player,Team,Minutes,Actions,Total xT,xT per 90
17,Dries Mertens,Belgium,232,273,1.270,0.490
23,Heung-Min Son,South Korea,228,227,1.070,0.420
22,Miguel Arturo Layún Prado,Mexico,249,270,1.090,0.390
30,M''Baye Babacar Niang,Senegal,245,211,0.990,0.360
40,Denis Cheryshev,Russia,214,213,0.850,0.360
45,Wahbi Khazri,Tunisia,208,184,0.790,0.340
58,Hirving Rodrigo Lozano Bahena,Mexico,181,288,0.660,0.330
59,Edgar Yoel Bárcenas Herrera,Panama,187,174,0.660,0.320
80,Ahmed Musa,Nigeria,202,161,0.520,0.230
82,Timo Werner,Germany,202,210,0.510,0.230


This view looks for players who add clear possession value without dominating action volume. Under the current filters, **Dries Mertens** is the strongest low-volume profile in the sample. That makes the section useful for identifying efficient contributors who might be missed by broader leaderboards built mainly on total output.

## ML Extension: Player Style Clustering and Similarity

Standard leaderboards show who produces value. Clustering adds a different layer by showing *how* that value is produced and which players share similar action-value profiles.

That is useful in practical football work because profiling and player identification are not always about finding the highest xT player in the dataset. Often the more relevant question is: which players generate value in a similar way, and what type of possession role do they represent?


### Features Used for the ML Analysis

The clustering model is built from a player feature table that combines `player_summary.parquet` and `actions.parquet`.

The features are intentionally interpretable.

- `total_xt` and `xt_per90` capture overall value and rate value from `player_summary.parquet`.
- `action_count` provides a basic activity measure from `player_summary.parquet`.
- `xt_per_action` estimates how much value a player adds each time they are involved.
- `progression_share` and `final_third_share` come from action-level aggregates in `actions.parquet` and describe where a player's threat is being generated.
- `pass_xt_share` measures how much of a player's xT comes through passing.
- `carry_dribble_xt_share` measures how much of a player's xT comes through carrying and dribble-style actions.

A minutes threshold is used before fitting the model because clustering becomes much less useful if the feature table is dominated by very short samples. The aim is to cluster real playing profiles rather than noise.


In [10]:
player_style_features = None
cluster_summary_table = None
style_outliers = None
cluster_model = None
cluster_feature_columns = None
scaled_player_features = None
cluster_explainer = None

if player_summary is None or actions is None:
    display(Markdown("The clustering section requires both `player_summary.parquet` and `actions.parquet`."))
else:
    import os

    for env_name in [
        "OMP_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "MKL_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
        "NUMEXPR_NUM_THREADS",
    ]:
        os.environ.setdefault(env_name, "1")

    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA

    cluster_minutes_threshold = 180
    cluster_count = 5

    ml_player_base = (
        player_summary.loc[:, ["player_name", "team_name", "minutes_played", "action_count", "total_xt", "xt_per90"]]
        .dropna()
        .copy()
    )
    ml_player_base = ml_player_base.loc[
        (ml_player_base["minutes_played"] >= cluster_minutes_threshold)
        & (ml_player_base["total_xt"] > 0)
        & (ml_player_base["action_count"] > 0)
    ].copy()

    ml_actions = actions.loc[:, ["player_name", "action_type", "xt_added", "progression_xt", "final_third_threat"]].copy()
    ml_actions["action_type"] = ml_actions["action_type"].fillna("unknown")
    for column in ["xt_added", "progression_xt", "final_third_threat"]:
        ml_actions[column] = pd.to_numeric(ml_actions[column], errors="coerce").fillna(0)

    ml_actions["pass_xt_component"] = ml_actions["xt_added"].where(ml_actions["action_type"].eq("pass"), 0)
    ml_actions["carry_dribble_xt_component"] = ml_actions["xt_added"].where(
        ml_actions["action_type"].isin(["carry", "dribble", "take_on"]),
        0,
    )

    action_feature_table = (
        ml_actions.groupby("player_name", as_index=False)
        .agg(
            progression_xt=("progression_xt", "sum"),
            final_third_threat=("final_third_threat", "sum"),
            pass_xt=("pass_xt_component", "sum"),
            carry_dribble_xt=("carry_dribble_xt_component", "sum"),
        )
    )

    player_style_features = ml_player_base.merge(action_feature_table, on="player_name", how="left").fillna(0)
    player_style_features["phase_xt_total"] = (
        player_style_features["progression_xt"] + player_style_features["final_third_threat"]
    )
    player_style_features = player_style_features.loc[player_style_features["phase_xt_total"] > 0].copy()

    player_style_features["xt_per_action"] = safe_divide(
        player_style_features["total_xt"],
        player_style_features["action_count"],
    )
    player_style_features["progression_share"] = safe_divide(
        player_style_features["progression_xt"],
        player_style_features["phase_xt_total"],
    )
    player_style_features["final_third_share"] = safe_divide(
        player_style_features["final_third_threat"],
        player_style_features["phase_xt_total"],
    )
    player_style_features["pass_xt_share"] = safe_divide(
        player_style_features["pass_xt"],
        player_style_features["total_xt"],
    )
    player_style_features["carry_dribble_xt_share"] = safe_divide(
        player_style_features["carry_dribble_xt"],
        player_style_features["total_xt"],
    )

    player_style_features = (
        player_style_features
        .replace([np.inf, -np.inf], pd.NA)
        .dropna(subset=[
            "total_xt",
            "xt_per90",
            "action_count",
            "xt_per_action",
            "progression_share",
            "final_third_share",
            "pass_xt_share",
            "carry_dribble_xt_share",
        ])
        .copy()
    )

    cluster_feature_columns = [
        "total_xt",
        "xt_per90",
        "action_count",
        "xt_per_action",
        "progression_share",
        "final_third_share",
        "pass_xt_share",
        "carry_dribble_xt_share",
    ]

    scaler = StandardScaler()
    scaled_player_features = scaler.fit_transform(player_style_features[cluster_feature_columns])

    cluster_model = KMeans(n_clusters=cluster_count, random_state=42, n_init=20)
    player_style_features["cluster"] = cluster_model.fit_predict(scaled_player_features)

    pca = PCA(n_components=2, random_state=42)
    pca_coordinates = pca.fit_transform(scaled_player_features)
    player_style_features["pca_1"] = pca_coordinates[:, 0]
    player_style_features["pca_2"] = pca_coordinates[:, 1]

    centroid_distances = np.linalg.norm(
        scaled_player_features - cluster_model.cluster_centers_[player_style_features["cluster"]],
        axis=1,
    )
    player_style_features["distance_to_cluster_center"] = centroid_distances

    cluster_summary_table = (
        player_style_features.groupby("cluster", as_index=False)[cluster_feature_columns]
        .mean()
        .round(3)
        .sort_values("xt_per90", ascending=False)
    )

    style_outliers = (
        player_style_features.loc[:, ["player_name", "team_name", "cluster", "distance_to_cluster_center"]]
        .sort_values("distance_to_cluster_center", ascending=False)
        .head(5)
        .rename(
            columns={
                "player_name": "Player",
                "team_name": "Team",
                "cluster": "Cluster",
                "distance_to_cluster_center": "Style Distance",
            }
        )
    )
    style_outliers["Style Distance"] = style_outliers["Style Distance"].round(2)

    display(cluster_summary_table)
    display(style_outliers)

    high_rate_cluster = int(cluster_summary_table.sort_values("xt_per90", ascending=False).iloc[0]["cluster"])
    high_pass_cluster = int(cluster_summary_table.sort_values("pass_xt_share", ascending=False).iloc[0]["cluster"])
    high_carry_cluster = int(cluster_summary_table.sort_values("carry_dribble_xt_share", ascending=False).iloc[0]["cluster"])

    cluster_explainer = f"""
The clustering model groups players by the shape of their possession-value profile rather than by total output alone. In this sample, **Cluster {high_rate_cluster}** carries the strongest rate value, **Cluster {high_pass_cluster}** is the most pass-driven, and **Cluster {high_carry_cluster}** leans most heavily on carry and dribble-led value. That is the main benefit of clustering here: it separates different *ways* of generating threat, not just different totals.
"""
    display(Markdown(cluster_explainer))


,cluster,total_xt,xt_per90,action_count,xt_per_action,progression_share,final_third_share,pass_xt_share,carry_dribble_xt_share
0,0,1.812,0.558,658.900,0.003,0.068,0.932,0.206,0.024
3,3,0.776,0.290,267.455,0.003,-0.011,1.011,0.156,0.060
1,1,0.363,0.132,307.759,0.001,0.103,0.897,0.354,0.018
2,2,0.225,0.089,350.500,0.001,1.267,-0.267,1.222,0.170
4,4,0.170,0.053,516.200,0.000,0.171,0.829,1.631,-0.403


,Player,Team,Cluster,Style Distance
37,José Andrés Guardado Hernández,Mexico,3,4.620
66,Romelu Lukaku Menama,Belgium,1,4.200
61,Artem Dzyuba,Russia,1,3.380
28,Jordan Brian Henderson,England,1,2.860
0,Kieran Trippier,England,0,2.750



The clustering model groups players by the shape of their possession-value profile rather than by total output alone. In this sample, **Cluster 0** carries the strongest rate value, **Cluster 4** is the most pass-driven, and **Cluster 2** leans most heavily on carry and dribble-led value. That is the main benefit of clustering here: it separates different *ways* of generating threat, not just different totals.


In [11]:
if player_style_features is None or player_style_features.empty:
    display(Markdown("Player-style clustering could not be produced from the currently loaded data."))
else:
    fig = px.scatter(
        player_style_features,
        x="pca_1",
        y="pca_2",
        color=player_style_features["cluster"].astype(str),
        hover_name="player_name",
        hover_data={
            "team_name": True,
            "total_xt": ":.2f",
            "xt_per90": ":.2f",
            "action_count": True,
            "cluster": True,
        },
        title="Player Style Clusters in PCA Space",
        template="plotly_white",
    )
    fig.update_layout(height=560, legend_title_text="Cluster")
    fig.update_xaxes(title="PCA 1")
    fig.update_yaxes(title="PCA 2")
    display(fig)

    outlier_row = style_outliers.iloc[0] if style_outliers is not None and not style_outliers.empty else None
    if outlier_row is not None:
        display(Markdown(
            f"The PCA view compresses the player feature set into a readable two-dimensional map of style similarity. "
            f"Players positioned close together have more comparable action-value profiles, while players further away are more stylistically distinct. "
            f"In the current sample, **{outlier_row['Player']}** appears among the strongest stylistic outliers, which suggests a profile that is less interchangeable with the rest of the player pool."
        ))


The PCA view compresses the player feature set into a readable two-dimensional map of style similarity. Players positioned close together have more comparable action-value profiles, while players further away are more stylistically distinct. In the current sample, **José Andrés Guardado Hernández** appears among the strongest stylistic outliers, which suggests a profile that is less interchangeable with the rest of the player pool.

In [12]:
if player_style_features is None or player_style_features.empty or scaled_player_features is None:
    display(Markdown("Player similarity could not be produced from the currently loaded data."))
else:
    from sklearn.neighbors import NearestNeighbors

    similarity_model = NearestNeighbors(n_neighbors=6, metric="euclidean")
    similarity_model.fit(scaled_player_features)

    example_players = []
    if top_player_table is not None and not top_player_table.empty:
        example_players.append(top_player_table.iloc[0]["Player"])
    if low_volume_table is not None and not low_volume_table.empty:
        example_players.append(low_volume_table.iloc[0]["Player"])

    seen = set()
    example_players = [name for name in example_players if not (name in seen or seen.add(name))][:2]

    if not example_players:
        display(Markdown("No example players are available for the similarity section."))
    else:
        for player_name in example_players:
            player_index = player_style_features.index[player_style_features["player_name"].eq(player_name)]
            if len(player_index) == 0:
                continue

            row_position = player_style_features.index.get_loc(player_index[0])
            distances, neighbor_indices = similarity_model.kneighbors(scaled_player_features[[row_position]])

            similarity_table = (
                player_style_features.iloc[neighbor_indices[0]]
                .loc[:, ["player_name", "team_name", "cluster"]]
                .copy()
            )
            similarity_table["Feature Distance"] = distances[0]
            similarity_table = similarity_table.iloc[1:].copy()
            similarity_table["Similarity Score"] = 1 / (1 + similarity_table["Feature Distance"])
            similarity_table = similarity_table.rename(
                columns={
                    "player_name": "Matched Player",
                    "team_name": "Team",
                    "cluster": "Cluster",
                }
            )
            similarity_table["Feature Distance"] = similarity_table["Feature Distance"].round(2)
            similarity_table["Similarity Score"] = similarity_table["Similarity Score"].round(2)

            display(Markdown(f"#### Stylistic matches for {player_name}"))
            display(similarity_table.reset_index(drop=True))

        display(Markdown(
            "Similarity analysis is not trying to find players with the same total xT number. It is looking for players whose feature profile is shaped in a similar way once the variables are standardized. That makes it more useful for profiling, shortlist building, and role-based comparison than a simple leaderboard rank."
        ))


#### Stylistic matches for Kieran Trippier

,Matched Player,Team,Cluster,Feature Distance,Similarity Score
0,Kevin De Bruyne,Belgium,0,2.190,0.310
1,Willian Borges da Silva,Brazil,0,2.810,0.260
2,Philippe Coutinho Correia,Brazil,0,3.420,0.230
3,Eden Hazard,Belgium,0,3.470,0.220
4,Luka Modrić,Croatia,0,3.490,0.220


#### Stylistic matches for Dries Mertens

,Matched Player,Team,Cluster,Feature Distance,Similarity Score
0,Heung-Min Son,South Korea,3,0.730,0.580
1,M''Baye Babacar Niang,Senegal,3,0.980,0.510
2,Miguel Arturo Layún Prado,Mexico,3,1.000,0.500
3,Denis Cheryshev,Russia,3,1.290,0.440
4,Wahbi Khazri,Tunisia,3,1.430,0.410


Similarity analysis is not trying to find players with the same total xT number. It is looking for players whose feature profile is shaped in a similar way once the variables are standardized. That makes it more useful for profiling, shortlist building, and role-based comparison than a simple leaderboard rank.

## Key Findings

The notebook closes with a short reporting summary rather than a long conclusion. The aim is to keep the final section practical and decision-focused.


In [13]:
key_findings = []

if top_player_table is not None and not top_player_table.empty:
    player_leader = top_player_table.iloc[0]
    key_findings.append(
        f"**Player value:** {player_leader['Player']} leads the qualified sample on total xT, making them the clearest example of sustained possession value over meaningful involvement."
    )

if action_type_chart_df is not None and not action_type_chart_df.empty:
    top_action = action_type_chart_df.iloc[0]
    key_findings.append(
        f"**Action type:** {top_action['type_name']} is the strongest aggregate xT contributor, which reinforces that threat is being driven by a specific subset of on-ball actions rather than by generic touch volume."
    )

if team_bar_df is not None and not team_bar_df.empty and team_scatter_df is not None and not team_scatter_df.empty:
    top_team = team_bar_df.iloc[0]
    top_progression_team = team_scatter_df.sort_values("progression_xt", ascending=False).iloc[0]
    key_findings.append(
        f"**Team style:** {top_team['team_name']} leads on xT per match, while {top_progression_team['team_name']} stands out most strongly for progression-led threat, showing that total value and route-to-value are not always the same thing."
    )

if low_volume_table is not None and not low_volume_table.empty:
    low_volume_leader = low_volume_table.iloc[0]
    key_findings.append(
        f"**Low-volume profile:** {low_volume_leader['Player']} stands out once minutes and action-count filters are tightened, which suggests meaningful value can still appear outside the highest-volume player group."
    )

if cluster_summary_table is not None and not cluster_summary_table.empty:
    top_cluster = int(cluster_summary_table.sort_values("xt_per90", ascending=False).iloc[0]["cluster"])
    key_findings.append(
        f"**ML extension:** The clustering work separates players into distinct action-value profiles rather than a single ranked list, and Cluster {top_cluster} currently represents the strongest rate-value archetype in the sample."
    )

if key_findings:
    display(Markdown("\n".join(f"- {item}" for item in key_findings)))
else:
    display(Markdown("Processed data is not available in a complete enough state to support a findings summary."))


- **Player value:** Kieran Trippier leads the qualified sample on total xT, making them the clearest example of sustained possession value over meaningful involvement.
- **Action type:** shot is the strongest aggregate xT contributor, which reinforces that threat is being driven by a specific subset of on-ball actions rather than by generic touch volume.
- **Team style:** Germany leads on xT per match, while England stands out most strongly for progression-led threat, showing that total value and route-to-value are not always the same thing.
- **Low-volume profile:** Dries Mertens stands out once minutes and action-count filters are tightened, which suggests meaningful value can still appear outside the highest-volume player group.
- **ML extension:** The clustering work separates players into distinct action-value profiles rather than a single ranked list, and Cluster 0 currently represents the strongest rate-value archetype in the sample.

## Next Steps

There are several practical ways to extend this project without changing the central possession-value framing.

- broaden the sample to more competitions so style and similarity results are less tournament-specific
- add role archetypes so midfielders, wide players, and full-backs are compared in better context
- expand the similarity search into a broader scouting workflow with shortlist-style outputs
- test richer action-value frameworks beyond xT while keeping the same reporting structure
- connect the style clusters to recruitment and succession-planning use cases

The current notebook is designed as a strong first reporting layer. The next stage would be to turn that reporting layer into a more role-aware scouting and profiling tool.
